# VoltRelay Energy — What is really driving service quality, retention and margin?

**Gradient Learnings Data Analytics Hackathon** · Battery-swap network, 6 Indian cities, Jan 2024 – Jun 2025

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andringodson/Hackathon-DataAnalytics/blob/main/notebooks/VoltRelay_Analysis.ipynb)

This notebook runs top to bottom in Google Colab. The first code cell downloads the eight organiser CSVs (~830 MB) from Google Drive.

| Section | Contents |
|---|---|
| 1 | Setup and data loading |
| 2 | Data understanding |
| 3 | Data cleaning — every documented quality issue tested, plus three undocumented ones |
| 4 | Metric definitions and unit-economics model |
| 5–10 | The six Core Questions |
| 11 | Optional deep dives: budget decision, expansion waves, partner value, ticket text, anomalies |
| 12 | Key findings and recommendations |
| 13 | Export of aggregated tables for the dashboard |

## 1. Setup and data loading

In [ ]:
import os, sys, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
try:
    import gdown  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

DATA_DIR = Path(os.environ.get("VOLTRELAY_DATA", "data/raw"))
FIG_DIR = Path(os.environ.get("VOLTRELAY_FIGS", "figures"))
OUT_DIR = Path(os.environ.get("VOLTRELAY_OUT", "processed"))
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TABLES = ["swap_events", "station_hourly_status", "riders", "batteries", "support_tickets", "stations", "city_daily_context", "fleet_partners"]
DRIVE_FOLDER = "https://drive.google.com/drive/folders/1qsjGWrgmEvOaf2Is2fh7RgVEkKd1V3df"


def find(name):
    hits = sorted(DATA_DIR.rglob(f"{name}.csv*"))
    return hits[0] if hits else None


if any(find(t) is None for t in TABLES):
    import gdown
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(DRIVE_FOLDER, output=str(DATA_DIR), quiet=True, remaining_ok=True)
missing = [t for t in TABLES if find(t) is None]
assert not missing, f"Missing files: {missing}"
print({t: f"{find(t).stat().st_size / 1e6:,.1f} MB" for t in TABLES})

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
PAL = {"primary": "#1f5f8b", "accent": "#e07a1f", "bad": "#c0392b", "good": "#2e8b57", "muted": "#8c8c8c"}
GEN_PAL = {"Gen1": "#c0392b", "Gen2": "#1f5f8b", "Gen3": "#2e8b57"}
plt.rcParams.update({"figure.dpi": 90, "savefig.dpi": 130, "axes.titleweight": "bold", "axes.titlesize": 12})


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()


def pct(ax, axis="y", decimals=0):
    fmt = mtick.PercentFormatter(1.0, decimals=decimals)
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(fmt)

In [ ]:
cat = "category"
se = pd.read_csv(find("swap_events"), parse_dates=["event_ts"], dtype={
    "rider_id": cat, "station_id": cat, "event_type": cat, "battery_in_id": cat, "battery_out_id": cat,
    "tariff_code": cat, "payment_mode": cat, "station_firmware": cat, "sync_mode": cat})
hs = pd.read_csv(find("station_hourly_status"), parse_dates=["hour_start"], dtype={"station_id": cat, "telemetry_status": cat})
rd = pd.read_csv(find("riders"), parse_dates=["signup_date"])
bt = pd.read_csv(find("batteries"), parse_dates=["manufacture_date", "commission_date", "retired_date"])
tk = pd.read_csv(find("support_tickets"), parse_dates=["created_ts"])
st = pd.read_csv(find("stations"), parse_dates=["commissioned_date", "decommissioned_date", "firmware_updated_date", "competitor_within_1_5km_since"])
cx = pd.read_csv(find("city_daily_context"), parse_dates=["date"])
fp = pd.read_csv(find("fleet_partners"), parse_dates=["contract_start_date", "amendment_date"])
raw = {"swap_events": se, "station_hourly_status": hs, "riders": rd, "batteries": bt, "support_tickets": tk,
       "stations": st, "city_daily_context": cx, "fleet_partners": fp}
pd.DataFrame({k: {"rows": len(v), "columns": v.shape[1]} for k, v in raw.items()}).T

## 2. Data understanding

The core fact table is `swap_events` — one row per swap *attempt*, completed or not. Everything else hangs off it:

```
fleet_partners -> riders -> swap_events -> stations -> station_hourly_status
                                 |   \-> batteries (battery_in_id, battery_out_id)
                                 \-> support_tickets -> stations / batteries (optional)
stations.city + date -> city_daily_context
```

In [ ]:
se.head(3).T

In [ ]:
print("Event outcomes:")
display(se.event_type.value_counts(normalize=True).rename("share").to_frame())
print("Tariff codes:", se.tariff_code.value_counts().to_dict())
print("Payment modes:", se.payment_mode.value_counts().to_dict())
print("Date range:", se.event_ts.min(), "->", se.event_ts.max())

**Referential integrity.** Every foreign key resolves:

In [ ]:
checks = {
    "swap_events.rider_id in riders": se.rider_id.isin(rd.rider_id).mean(),
    "swap_events.station_id in stations": se.station_id.isin(st.station_id).mean(),
    "swap_events.battery_out_id in batteries": se.battery_out_id.dropna().isin(bt.battery_id).mean(),
    "support_tickets.rider_id in riders": tk.rider_id.isin(rd.rider_id).mean(),
    "riders.partner_id in fleet_partners": rd.partner_id.dropna().isin(fp.partner_id).mean(),
}
pd.Series(checks, name="share matched").to_frame()

**Which fields are populated for which outcome.** Pricing, energy and outgoing-battery fields exist only for completed swaps; non-completed attempts carry ₹0. Revenue and unit-economics analysis therefore uses completed swaps only, while service-quality analysis uses all attempts.

In [ ]:
se.groupby("event_type", observed=True)[["battery_in_id", "battery_out_id", "km_since_last_swap", "energy_to_recharge_kwh", "amount_charged_inr"]].agg(
    lambda s: s.notna().mean() if s.name != "amount_charged_inr" else s.mean()).rename(columns={"amount_charged_inr": "mean ₹ charged"})

## 3. Data cleaning

Each documented data-quality issue is **detected with evidence, then handled deliberately**. The analysis also found three undocumented issues (3.8).

### 3.1 Test stations — excluded, but they are not "small"
The brief describes `STN-TST-*` as inactive test stations with a few zero-value transactions. In this export they are anything but.

In [ ]:
tst_ids = st.loc[st.station_id.str.startswith("STN-TST"), "station_id"].tolist()
tst = se[se.station_id.isin(tst_ids)]
real_avg = se[~se.station_id.isin(tst_ids)].groupby("station_id", observed=True).size().mean()
display(tst.groupby("station_id", observed=True).agg(events=("event_id", "size"), revenue_inr=("amount_charged_inr", "sum"),
        zero_value_share=("amount_charged_inr", lambda s: (s == 0).mean()), first=("event_ts", "min"), last=("event_ts", "max")))
print(f"Average real station: {real_avg:,.0f} events")

**Decision:** exclude both test stations (62K events, ₹3.9M) from every network metric, as the brief instructs. **Flag:** the test stations handle more traffic than an average real station and book real revenue for all 18 months. Either they are live sites that were mislabelled, or revenue is being recorded on test infrastructure. Finance should reconcile this.

In [ ]:
se = se[~se.station_id.isin(tst_ids)].copy()
hs = hs[~hs.station_id.isin(tst_ids)].copy()
tk = tk[~tk.station_id.isin(tst_ids)].copy()
st = st[~st.station_id.isin(tst_ids)].copy()
for df in (se, hs):
    df["station_id"] = df.station_id.cat.remove_unused_categories()
print("Stations remaining:", st.shape[0])

### 3.2 Firmware timezone bug (v3.2.0, 10 Mar – 14 Apr 2025)
The strongest proof: the **PEAK tariff only applies at 12–14h and 19–22h**, yet affected events show PEAK prices charged at 06–08h and 13–16h. That is exactly 5 h 30 m earlier than the true time (IST vs UTC).

In [ ]:
bug = (se.station_firmware == "v3.2.0") & (se.event_ts >= "2025-03-10") & (se.event_ts < "2025-04-15")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for ax, (label, mask) in zip(axes, [("v3.2.0 during bug window", bug), ("All other events", ~bug)]):
    se.loc[mask, "event_ts"].dt.hour.value_counts(normalize=True).sort_index().plot.bar(ax=ax, color=PAL["bad"] if "bug" in label else PAL["primary"])
    ax.set_title(label); ax.set_xlabel("logged hour of day"); pct(ax)
axes[0].set_ylabel("share of events")
save(fig, "03_firmware_bug_hours")
peak_hours_logged = se.loc[bug & (se.tariff_code == "PEAK"), "event_ts"].dt.hour.value_counts().sort_index()
print("PEAK-tariff events by *logged* hour in the bug window:", peak_hours_logged[peak_hours_logged > 0].to_dict())

In [ ]:
se["ts_corrected"] = bug
se.loc[bug, "event_ts"] = se.loc[bug, "event_ts"] + pd.Timedelta(hours=5, minutes=30)
chk = se.loc[bug & (se.tariff_code == "PEAK"), "event_ts"].dt.hour.value_counts()
print(f"Corrected {bug.sum():,} timestamps. PEAK events now fall in hours: {sorted(chk[chk > 0].index.tolist())}")

### 3.3 Offline-sync near-duplicates — tested, none present
Definition from the brief: same rider, station and batteries, a few seconds to two minutes apart, concentrated in `offline_batch` rows at poor-connectivity stations.

In [ ]:
s = se.sort_values(["rider_id", "station_id", "event_ts"])
p = s.shift()
gap = (s.event_ts - p.event_ts).dt.total_seconds()
same_rs = s.rider_id.astype(str).eq(p.rider_id.astype(str)) & s.station_id.astype(str).eq(p.station_id.astype(str))
same_batt = (s.battery_in_id.astype(str).eq(p.battery_in_id.astype(str)) & s.battery_out_id.astype(str).eq(p.battery_out_id.astype(str)))
exact_dups = same_rs & same_batt & (gap <= 120)
close = same_rs & (gap <= 120)
print(f"Exact near-duplicates (same rider+station+batteries, <=2 min): {exact_dups.sum()}")
print(f"Same rider+station within 2 min (any batteries): {close.sum():,}")
print(f"  offline_batch share among them: {(s.sync_mode[close] == 'offline_batch').mean():.1%}  vs network baseline {(se.sync_mode == 'offline_batch').mean():.1%}")
conn = se[["station_id", "sync_mode"]].merge(st[["station_id", "connectivity_tier"]], on="station_id")
print(pd.crosstab(conn.connectivity_tier, conn.sync_mode, normalize="index").round(3))
del s, p, gap

**Decision:** no rows removed. Close same-rider pairs always involve *different* battery packs, and `offline_batch` is no more common among them than in the network overall. They are genuine back-to-back attempts, not sync retries. The check stays in the pipeline so a future export with real duplicates would be caught. `offline_batch` itself does behave as documented: it only occurs at fair and poor-connectivity stations.

### 3.4 Distance and charge-reading outliers

In [ ]:
km = se.km_since_last_swap
print(km.describe(percentiles=[.001, .01, .99, .999]).round(1).to_dict())
km_bad = (km < 0) | (km > 250)
print(f"Negative (odometer reset): {(km < 0).sum():,} | >250 km (implausible for a 2.1/4.8 kWh pack): {(km > 250).sum():,}")
se["km_valid"] = km.where(~km_bad)
for c in ["soc_in_pct", "soh_in_pct", "soc_out_pct", "soh_out_pct"]:
    n = (se[c] > 100).sum()
    se[c] = se[c].clip(upper=100)
    print(f"{c}: {n:,} readings >100% clipped to 100 (sensor drift)")

### 3.5 Inconsistent city spellings in `riders.home_city`

In [ ]:
city_map = {"bengaluru": "Bengaluru", "bangalore": "Bengaluru", "blr": "Bengaluru",
            "delhi ncr": "Delhi NCR", "delhi": "Delhi NCR", "new delhi": "Delhi NCR", "gurgaon": "Delhi NCR",
            "hyderabad": "Hyderabad", "hyd": "Hyderabad", "pune": "Pune", "pun": "Pune",
            "mumbai": "Mumbai", "bombay": "Mumbai", "mum": "Mumbai", "jaipur": "Jaipur", "jai": "Jaipur"}
print("Raw variants:", rd.home_city.nunique())
rd["home_city"] = rd.home_city.str.strip().str.lower().map(city_map)
assert rd.home_city.notna().all()
rd.home_city.value_counts().to_frame().T

### 3.6 Missing telemetry — kept as missing, never zero
Blank telemetry fields line up exactly with `telemetry_status` ∈ {partial, missing}, concentrated at poor-connectivity stations. Hourly aggregates use only `ok` rows, so a blank is never read as "0 charged packs" (which would fake a stockout).

In [ ]:
hs = hs.merge(st[["station_id", "connectivity_tier", "charger_generation", "city", "expansion_wave"]], on="station_id")
display(pd.crosstab(hs.connectivity_tier, hs.telemetry_status, normalize="index").round(3))
print("Null charged_2w_min where status == ok:", hs.loc[hs.telemetry_status == "ok", "charged_2w_min"].isna().sum())

### 3.7 CSAT is missing not-at-random
CSAT exists **only for resolved tickets**, and half as often when resolution takes more than 24 h. A naive average therefore describes the happiest subset only. We report CSAT with its coverage and never as a stand-alone satisfaction metric.

In [ ]:
display(tk.groupby("resolution_status").csat_score.agg(coverage=lambda s: s.notna().mean(), mean="mean"))
display(tk.groupby(pd.cut(tk.resolution_hours, [0, 4, 12, 24, 48, 1000]), observed=True).csat_score.agg(coverage=lambda s: s.notna().mean(), mean="mean"))

### 3.8 Three undocumented issues found during profiling
1. **`payment_mode` is always `partner_invoice`**, even for independent pay-as-you-go riders. It carries no information, so rider segments come from `riders.plan_type` and `partner_id`.
2. **Tariffs `PREPAID` and `PROMO_FREE` never occur.** Prepaid riders are billed on `STD`/`PEAK`/`OFFPEAK`.
3. **Battery IDs are not physically consistent across events.** The same pack is "issued" in different cities minutes apart (below). Batteries are therefore analysed as *cohorts* (supplier, lot, SoH at swap time), not by tracing individual packs through the event log.

In [ ]:
b = se.dropna(subset=["battery_out_id"]).sort_values(["battery_out_id", "event_ts"])[["battery_out_id", "event_ts", "station_id", "rider_id"]]
bp = b.shift()
impossible = b.battery_out_id.astype(str).eq(bp.battery_out_id.astype(str)) & ((b.event_ts - bp.event_ts).dt.total_seconds() < 900) & b.station_id.astype(str).ne(bp.station_id.astype(str))
print(f"Same pack issued at two different stations <15 min apart: {impossible.sum():,} times")
display(pd.concat([bp[impossible].head(2), b[impossible].head(2)]).sort_values(["battery_out_id", "event_ts"]))
del b, bp

### 3.9 Cleaning summary

In [ ]:
pd.DataFrame([
    ["Test stations STN-TST-01/02", "62,031 events, ₹3.9M", "Excluded; flagged as anomaly"],
    ["Firmware v3.2.0 timezone bug", f"{bug.sum():,} events", "+5h30m correction"],
    ["Offline-sync near-duplicates", "0 found", "Check retained; nothing removed"],
    ["km_since_last_swap <0 or >250", f"{km_bad.sum():,} values", "Set to missing for range analysis"],
    ["SoC/SoH > 100%", "~13K readings", "Clipped to 100"],
    ["home_city spellings", "21 variants", "Mapped to 6 cities"],
    ["Missing telemetry", "46K hours", "Excluded from averages, never zero-filled"],
    ["CSAT MNAR", "66% missing", "Reported with coverage only"],
    ["Ticket categories", "'other' = vehicle complaints", "Re-classified with text rules (§11.4)"],
], columns=["Issue", "Scale", "Treatment"])

## 4. Metric definitions and unit-economics model

| Metric | Definition |
|---|---|
| **Attempt** | Any `swap_events` row |
| **Completed swap** | `event_type == swap_completed` |
| **Service failure** | `failed_no_charged_battery`, `failed_system_error` or `abandoned_queue`. `cancelled_by_rider` is excluded: those riders leave within 90 s, before the station has failed them. |
| **Failure rate** | Service failures ÷ attempts |
| **Revenue** | `amount_charged_inr` on completed swaps (non-completed attempts are ₹0) |
| **Contribution before wear (CM1)** | Revenue − energy cost − station fixed cost |
| **Contribution after wear (CM2)** | CM1 − battery wear cost |

**Cost model (per completed swap).** The brief names the three costs to include: energy, battery wear and station costs.
* **Energy** = `energy_to_recharge_kwh` × the station's `grid_tariff_inr_kwh`.
* **Station fixed cost** = the station's (monthly rent + maintenance) ÷ its completed swaps that month.
* **Battery wear** = pack purchase cost × the share of its useful life (initial SoH → 70% retirement threshold) consumed per swap. SoH lost *inside* the observation window is estimated by prorating lifetime loss by days in service within the window, then divided by the number of times the pack was issued. Fast-degrading packs therefore carry a proportionally higher wear cost.

Battery wear rests on the most assumptions, so every margin chart shows CM1 and CM2 side by side. The conclusions below hold under both.

In [ ]:
se = se.merge(st[["station_id", "city", "zone", "grid_tariff_inr_kwh", "charger_generation", "location_type", "expansion_wave",
                  "connectivity_tier", "host_type", "monthly_rent_inr", "monthly_maintenance_inr"]], on="station_id", how="left")
se = se.merge(rd[["rider_id", "vehicle_class", "partner_id", "plan_type"]], on="rider_id", how="left")
se["date"] = se.event_ts.dt.normalize()
se["month"] = se.event_ts.dt.to_period("M").dt.to_timestamp()
se["hour"] = se.event_ts.dt.hour
se["completed"] = se.event_type.eq("swap_completed")
se["service_failure"] = se.event_type.isin(["failed_no_charged_battery", "failed_system_error", "abandoned_queue"])
se["no_charge"] = se.event_type.eq("failed_no_charged_battery")

issues = se.loc[se.completed, "battery_out_id"].astype(str).value_counts().rename("issues")
bt = bt.merge(issues, left_on="battery_id", right_index=True, how="left")
bt["soh_loss"] = (bt.initial_soh_pct - bt.current_soh_pct).clip(lower=0)
end = bt.retired_date.fillna(pd.Timestamp("2025-06-30"))
life_days = (end - bt.commission_date).dt.days.clip(lower=1)
window_days = (end - bt.commission_date.clip(lower=pd.Timestamp("2024-01-01"))).dt.days.clip(lower=0)
bt["soh_loss_window"] = bt.soh_loss * window_days / life_days
bt["soh_loss_per_100_swaps"] = 100 * bt.soh_loss_window / bt.issues
bt["wear_inr_per_swap"] = bt.purchase_cost_inr * (bt.soh_loss_window / (bt.initial_soh_pct - 70)) / bt.issues

wear = bt.set_index("battery_id").wear_inr_per_swap
se["wear_inr"] = se.battery_out_id.astype(str).map(wear).where(se.completed)
se["energy_cost_inr"] = se.energy_to_recharge_kwh * se.grid_tariff_inr_kwh
sm = se[se.completed].groupby(["station_id", "month"], observed=True).size().rename("n").reset_index()
sm = sm.merge(st[["station_id", "monthly_rent_inr", "monthly_maintenance_inr"]], on="station_id")
sm["fixed_per_swap"] = (sm.monthly_rent_inr + sm.monthly_maintenance_inr) / sm.n
se = se.merge(sm[["station_id", "month", "fixed_per_swap"]], on=["station_id", "month"], how="left")
se.loc[~se.completed, "fixed_per_swap"] = np.nan
se["cm1"] = (se.amount_charged_inr - se.energy_cost_inr - se.fixed_per_swap).where(se.completed)
se["cm2"] = (se.cm1 - se.wear_inr).where(se.completed)
comp = se[se.completed]
comp[["amount_charged_inr", "energy_cost_inr", "fixed_per_swap", "wear_inr", "cm1", "cm2"]].describe().T[["mean", "25%", "50%", "75%"]]

## 5. Core Question 1 — Network performance over time
*How have completed swaps, revenue, failure rates and contribution margin per swap trended? Do they tell the same story?*

In [ ]:
m = se.groupby("month").agg(attempts=("event_id", "size"), completed=("completed", "sum"), failures=("service_failure", "sum"),
                            revenue=("amount_charged_inr", "sum"), stations=("station_id", "nunique"), riders=("rider_id", "nunique"))
m["failure_rate"] = m.failures / m.attempts
m = m.join(comp.groupby("month")[["amount_charged_inr", "energy_cost_inr", "fixed_per_swap", "wear_inr", "cm1", "cm2"]].mean().add_prefix("per_swap_"))
monthly = m.copy()
m[["attempts", "completed", "failures", "failure_rate", "revenue", "stations", "per_swap_amount_charged_inr", "per_swap_cm1", "per_swap_cm2"]]

In [ ]:
first, last = m.iloc[:3].mean(), m.iloc[-3:].mean()
growth = pd.DataFrame({"Q1 2024 (monthly avg)": first, "Q2 2025 (monthly avg)": last})
growth["change"] = growth.iloc[:, 1] / growth.iloc[:, 0] - 1
growth.loc[["completed", "revenue", "failures", "failure_rate", "riders", "stations", "per_swap_amount_charged_inr", "per_swap_cm1", "per_swap_cm2"]]

In [ ]:
EVENTS = [("2024-07-01", "price rise"), ("2024-08-01", "Kyron lots KY-2407..09"), ("2024-10-01", "peak pilot"), ("2024-11-01", "ZipDrop 12%→28%")]


def mark(ax, events=EVENTS, top=0.60, step=0.07):
    tr = ax.get_xaxis_transform()
    for i, (d, lbl) in enumerate(events):
        ax.axvline(pd.Timestamp(d), color=PAL["muted"], ls=":", lw=1)
        ax.text(pd.Timestamp(d), top - i * step, " " + lbl, transform=tr, fontsize=8, color="#444", va="top", ha="left",
                bbox=dict(facecolor="white", edgecolor="none", alpha=.8, pad=1))


fig, axes = plt.subplots(2, 2, figsize=(15, 8.5))
idx = m[["completed", "revenue", "failures"]] / m[["completed", "revenue", "failures"]].iloc[0] * 100
ax = axes[0, 0]
idx.plot(ax=ax, color=[PAL["primary"], PAL["good"], PAL["bad"]], lw=2.2)
ax.set_title("Growth index (Jan 2024 = 100): failures grew fastest"); ax.set_xlabel("")
ax = axes[0, 1]
ax.plot(m.index, m.failure_rate, color=PAL["bad"], lw=2.2, marker="o", ms=4)
ax.set_title("Service-failure rate: flat ~4% except two summer spikes"); pct(ax, decimals=0)
ax = axes[1, 0]
ax.bar(m.index, m.revenue / 1e6, width=20, color=PAL["good"])
ax.set_title("Monthly revenue (₹ million)")
ax = axes[1, 1]
ax.plot(m.index, m.per_swap_cm1, color=PAL["primary"], lw=2.2, marker="o", ms=4, label="CM1: before battery wear")
ax.plot(m.index, m.per_swap_cm2, color=PAL["accent"], lw=2.2, marker="o", ms=4, label="CM2: after battery wear")
ax.axhline(0, color="black", lw=.8)
ax.set_title("Contribution margin per completed swap (₹)"); ax.legend(loc="center left", fontsize=8); mark(ax)
fig.tight_layout()
save(fig, "05_network_trends")

In [ ]:
comp_cols = ["per_swap_amount_charged_inr", "per_swap_energy_cost_inr", "per_swap_fixed_per_swap", "per_swap_wear_inr"]
w = m[comp_cols].rename(columns=dict(zip(comp_cols, ["Revenue", "Energy", "Station fixed", "Battery wear"])))
fig, ax = plt.subplots(figsize=(13, 4.2))
ax.stackplot(w.index, w["Energy"], w["Station fixed"], w["Battery wear"], labels=["Energy", "Station fixed", "Battery wear"],
             colors=["#f2c14e", "#9aa5b1", PAL["accent"]], alpha=.85)
ax.plot(w.index, w["Revenue"], color=PAL["good"], lw=2.5, label="Revenue per swap")
ax.set_title("Per-swap revenue vs cost stack (₹): the price rise was absorbed by battery wear within two months")
ax.legend(loc="upper left", ncol=2, fontsize=8); ax.set_ylim(0, 125); mark(ax, top=0.97)
save(fig, "05_cost_stack")
w.round(2).iloc[[0, 5, 6, 8, 9, 10, 17]]

### Answer to Q1
**No — the metrics tell four different stories.**
* **Volume and revenue**: completed swaps grew **2.8×** and revenue **3.0×** between Q1 2024 and Q2 2025, as stations grew from 90 to 150 and monthly active riders grew 2.6×.
* **Failures grew faster than volume.** Monthly service failures rose **5.7×** (vs 2.8× for completions). The rate is not drifting up steadily: it sits at ~4% most of the year and **spikes every summer**, to 11.9% in May 2024 and 9.5% in May 2025. Growth simply puts more swaps into those summer months.
* **Margin moved against revenue.** The July 2024 base-price rise lifted revenue per swap by ~₹5 and briefly more than halved the per-swap loss (CM2 −₹14 → −₹7). From September 2024, **battery wear per swap jumped ~₹16 (+44%)** and the ZipDrop renegotiation (Nov 2024) added ~₹2 of network-wide discount per swap. Together these wiped out the price rise, and CM2 has sat near −₹16 to −₹20 since.
* Energy and station fixed costs per swap actually **fell** (better station utilisation; lower energy per swap as packs age). Battery wear is the cost that moved the margin.

## 6. Core Question 2 — Service failures and customer experience
*How do queue wait, failed and abandoned swaps relate to station, hour, season and vehicle class? Is service quality even, or concentrated?*

In [ ]:
fail_mix = pd.crosstab(se.month, se.event_type, normalize="index")
fig, ax = plt.subplots(figsize=(13, 3.8))
fail_mix.drop(columns="swap_completed").plot.area(ax=ax, color=["#e07a1f", "#9aa5b1", "#c0392b", "#6c3483"], alpha=.85)
ax.set_title("Non-completed attempts by type: summer spikes are stockouts ('no charged battery') plus the abandonment they cause")
pct(ax, decimals=0); ax.set_xlabel(""); ax.legend(fontsize=8, loc="upper center", ncol=4)
save(fig, "06_failure_mix")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
h = se.groupby("hour").agg(fail=("service_failure", "mean"), attempts=("event_id", "size"))
ax = axes[0]
ax.bar(h.index, h.attempts / 1e3, color="#d5dde5")
ax2 = ax.twinx(); ax2.plot(h.index, h.fail, color=PAL["bad"], marker="o", lw=2); pct(ax2, decimals=1); ax2.grid(False)
ax.set_title("By hour: evening peak fails most"); ax.set_xlabel("hour (corrected)"); ax.set_ylabel("attempts (000s)"); ax2.set_ylabel("failure rate")
mv = se.pivot_table(index="month", columns="vehicle_class", values="service_failure", aggfunc="mean")
ax = axes[1]
mv.plot(ax=ax, color=[PAL["primary"], PAL["accent"]], lw=2.2, marker="o", ms=3)
ax.set_title("By vehicle class: 3W fails 2× as often"); pct(ax, decimals=0); ax.set_xlabel("")
city_m = se.pivot_table(index="city", columns="month", values="service_failure", aggfunc="mean")
ax = axes[2]
sns.heatmap(city_m, ax=ax, cmap="Reds", cbar_kws={"format": mtick.PercentFormatter(1.0, decimals=0)}, xticklabels=3)
ax.set_xticklabels([pd.Timestamp(t.get_text()).strftime("%b-%y") for t in ax.get_xticklabels()], rotation=45)
ax.set_title("By city × month: summer spikes hit Delhi, Jaipur, Hyderabad"); ax.set_xlabel(""); ax.set_ylabel("")
fig.tight_layout()
save(fig, "06_failure_hour_class_city")

In [ ]:
se["season"] = se.month.dt.month.map({12: "Winter (Dec-Feb)", 1: "Winter (Dec-Feb)", 2: "Winter (Dec-Feb)", 3: "Summer (Mar-May)", 4: "Summer (Mar-May)",
                                      5: "Summer (Mar-May)", 6: "Monsoon (Jun-Sep)", 7: "Monsoon (Jun-Sep)", 8: "Monsoon (Jun-Sep)", 9: "Monsoon (Jun-Sep)",
                                      10: "Post-monsoon (Oct-Nov)", 11: "Post-monsoon (Oct-Nov)"})
display(se.groupby(["season", "vehicle_class"]).service_failure.mean().unstack().style.format("{:.1%}"))

**Queue wait is not where the experience breaks.** Mean wait for completed swaps is ~243 s in every hour, city and station generation, with no link to load. Failure shows up as *no battery at all*, not as a longer queue. Abandoned attempts are the exception: riders give up after a median ~11 minutes.

In [ ]:
display(se[se.completed].groupby("charger_generation").queue_wait_sec.describe()[["mean", "50%", "75%"]])
display(se.groupby("event_type", observed=True).queue_wait_sec.median().rename("median wait (s)").to_frame().T)
print("Correlation of station-day mean wait with station-day failure rate:",
      round(se.groupby(["station_id", "date"], observed=True).agg(w=("queue_wait_sec", "mean"), f=("service_failure", "mean")).corr().iloc[0, 1], 3))

### Is failure concentrated?

In [ ]:
sf = se.groupby("station_id", observed=True).agg(attempts=("event_id", "size"), failures=("service_failure", "sum"), fail=("service_failure", "mean")).join(
    st.set_index("station_id")[["city", "charger_generation", "location_type", "expansion_wave", "connectivity_tier"]])
sf = sf.sort_values("fail", ascending=False)
lor = sf.sort_values("failures", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.3))
ax = axes[0]
ax.plot(np.arange(1, len(lor) + 1) / len(lor), lor.failures.cumsum() / lor.failures.sum(), color=PAL["bad"], lw=2.2, label="failures")
ax.plot(np.arange(1, len(lor) + 1) / len(lor), lor.attempts.cumsum() / lor.attempts.sum(), color=PAL["primary"], lw=2.2, label="attempts (same stations)")
ax.plot([0, 1], [0, 1], color=PAL["muted"], ls=":")
pct(ax); pct(ax, "x"); ax.legend(); ax.set_xlabel("share of stations (worst first)"); ax.set_title("Top 20% of stations: 43% of failures, 31% of attempts")
ax = axes[1]
sns.stripplot(data=sf.reset_index(), x="city", y="fail", hue="charger_generation", palette=GEN_PAL, ax=ax, size=6, jitter=.25,
              order=["Jaipur", "Delhi NCR", "Hyderabad", "Pune", "Bengaluru", "Mumbai"])
pct(ax, decimals=0); ax.set_title("Station failure rate: the high tail is Gen1 in three cities"); ax.set_xlabel("")
save(fig, "06_station_concentration")
top = sf.head(25)
print(f"25 worst stations: {top.charger_generation.value_counts().to_dict()} | {top.expansion_wave.value_counts().to_dict()} | {top.city.value_counts().to_dict()}")
k = int(len(lor) * .2)
print(f"Top 20% of stations by failures hold {lor.failures.iloc[:k].sum() / lor.failures.sum():.1%} of failures and {lor.attempts.iloc[:k].sum() / lor.attempts.sum():.1%} of attempts")

In [ ]:
ct = pd.crosstab(se.charger_generation == "Gen1", se.service_failure)
chi2, pval, *_ = stats.chi2_contingency(ct)
g1 = se[se.charger_generation == "Gen1"].service_failure.mean()
g23 = se[se.charger_generation != "Gen1"].service_failure.mean()
print(f"Gen1 failure {g1:.2%} vs Gen2/3 {g23:.2%}  (chi-square={chi2:,.0f}, p={pval:.1e})")

### Answer to Q2
* **Not evenly spread.** Failures concentrate at **Gen1 charger stations in Delhi NCR, Jaipur and Hyderabad**. All 25 worst stations are Gen1 launch-era sites in those three cities (13 Delhi NCR, 10 Jaipur, 2 Hyderabad); the worst 20% of stations carry 43% of failures on 31% of attempts. Gen1 fails at 6.8% vs 4.4% for Gen2/3 (p ≈ 0).
* **Season beats hour.** Evening peaks (19–22 h) fail slightly more (6.2% vs ~5.0%), but the dominant pattern is **summer**: failure more than doubles from April to June, and the extra failures are stockouts plus the abandonment that follows.
* **3W riders fail twice as often as 2W riders** in every month (8–15% vs 3–11%). 3W slots are scarce, and 3W riders cannot fall back on 2W stock.
* **Queue wait is flat** (~4 minutes) and uninformative. The customer experience fails through *empty cabinets*, not slow queues.

## 7. Core Question 3 — Station and geographic patterns
*How do charger generation, location type and commissioning timing relate to service performance, battery turnaround time and complaints?*

In [ ]:
display(pd.crosstab(st.city, st.charger_generation, margins=True))
display(pd.crosstab(st.expansion_wave, [st.charger_generation]).join(pd.crosstab(st.expansion_wave, st.location_type)))

Gen1 cabinets are **only** in the Launch wave and are unevenly spread: 36 of the 51 are in Delhi NCR, Hyderabad and Jaipur, the three cities with the hottest summers (daily highs up to 46–49 °C). Mumbai has two.

### The mechanism: Gen1 cabinets cannot charge in the heat
Hourly telemetry (`ok` rows only) shows what happens inside the cabinet as ambient temperature rises.

In [ ]:
ok = hs[hs.telemetry_status == "ok"].copy()
ok["temp_band"] = pd.cut(ok.ambient_temp_c, [-10, 25, 30, 35, 40, 55], labels=["<25°C", "25–30", "30–35", "35–40", ">40°C"])
ok["stockout_2w"] = ok.charged_2w_min.eq(0)
tb = ok.groupby(["temp_band", "charger_generation"], observed=True).agg(charge_minutes=("avg_charge_minutes", "mean"), stockout_hours=("stockout_2w", "mean"),
                                                                       quarantined=("packs_quarantined", "mean"), hours=("station_id", "size")).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
sns.barplot(data=tb, x="temp_band", y="charge_minutes", hue="charger_generation", palette=GEN_PAL, ax=axes[0])
axes[0].set_title("Battery turnaround: Gen1 charge time doubles above 40°C"); axes[0].set_xlabel("ambient temperature"); axes[0].set_ylabel("avg charge minutes")
sns.barplot(data=tb, x="temp_band", y="stockout_hours", hue="charger_generation", palette=GEN_PAL, ax=axes[1])
axes[1].set_title("Share of hours with zero charged 2W packs"); axes[1].set_xlabel("ambient temperature"); pct(axes[1], decimals=0)
save(fig, "07_gen_heat_mechanism")
tb.pivot(index="temp_band", columns="charger_generation", values=["charge_minutes", "stockout_hours"]).round(3)

In [ ]:
d = se.groupby(["city", "date"], observed=True).agg(fail=("service_failure", "mean"), attempts=("event_id", "size")).reset_index().merge(cx, on=["city", "date"])
d["temp_band"] = pd.cut(d.max_temp_c, [0, 30, 35, 38, 41, 55], labels=["<30°C", "30–35", "35–38", "38–41", ">41°C"])
fig, ax = plt.subplots(figsize=(8, 3.6))
sns.barplot(data=d, x="temp_band", y="fail", color=PAL["bad"], ax=ax, errorbar=None)
pct(ax, decimals=0); ax.set_title("City-day failure rate by daily maximum temperature"); ax.set_xlabel("max temperature")
save(fig, "07_failure_vs_temp")
print(f"Heat-alert days: {d[d.heat_alert].fail.mean():.1%} vs other days {d[~d.heat_alert].fail.mean():.1%}")
print(f"Grid outage >1h days: {d[d.grid_outage_hours > 1].fail.mean():.1%} vs {d[d.grid_outage_hours <= 1].fail.mean():.1%}")
print(f"Flood-disruption days: {d[d.flood_disruption].fail.mean():.1%} (rain does not drive failures)")
print(d[["fail", "max_temp_c", "rainfall_mm", "grid_outage_hours"]].corr().fail.round(3).to_dict())

### Controlling for everything at once
Station-day OLS of failure rate on station attributes plus the day's max temperature, with the key **generation × heat** interaction. Standard errors are clustered by station.

In [ ]:
sd = se.groupby(["station_id", "date"], observed=True).agg(fail=("service_failure", "mean"), attempts=("event_id", "size")).reset_index()
sd = sd.merge(st[["station_id", "city", "charger_generation", "location_type", "expansion_wave", "connectivity_tier", "host_type"]], on="station_id")
sd = sd.merge(cx[["city", "date", "max_temp_c", "grid_outage_hours"]], on=["city", "date"])
sd["hot"] = (sd.max_temp_c > 38).astype(int)
ols = smf.wls("fail ~ C(charger_generation, Treatment('Gen2')) * hot + C(city) + C(location_type) + C(connectivity_tier) + C(host_type) + grid_outage_hours",
              data=sd, weights=sd.attempts).fit(cov_type="cluster", cov_kwds={"groups": sd.station_id.astype(str)})
coef = ols.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]]
coef[coef.index.str.contains("charger|hot|connectivity|location|grid|host")].round(4)

Holding city, site type, host and connectivity constant:
* On a normal day a Gen1 cabinet fails only **~0.4 pp** more often than Gen2.
* A >38 °C day adds **~5 pp** at a Gen2 station, and **a further ~9 pp at Gen1** (the interaction term), so Gen1 on a hot day is ~14 pp worse than a normal day. Gen3 largely cancels the heat effect (−3.7 pp).
* Each hour of city-wide grid outage adds ~0.7 pp.
* Location type, host type and connectivity are all within ±0.1 pp and mostly not significant.

The city differences are almost entirely "how many Gen1 cabinets does it have, and how hot does it get".

In [ ]:
tks = tk.dropna(subset=["station_id"]).merge(st[["station_id", "charger_generation", "city", "expansion_wave"]], on="station_id")
per1k = pd.DataFrame({col: (tks.groupby(col).size() / comp.groupby(col, observed=True).size() * 1000) for col in ["charger_generation"]})
city1k = (tks.groupby("city").size() / comp.groupby("city", observed=True).size() * 1000).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
city1k.plot.bar(ax=axes[0], color=PAL["primary"]); axes[0].set_title("Station-linked tickets per 1,000 completed swaps"); axes[0].set_xlabel("")
mix = pd.crosstab(tks.charger_generation, tks.category, normalize="index")
mix.plot.barh(stacked=True, ax=axes[1], colormap="tab20c"); axes[1].set_title("Ticket mix by charger generation"); pct(axes[1], "x")
axes[1].legend(fontsize=7, loc="lower center", bbox_to_anchor=(.5, -.45), ncol=3); axes[1].set_ylabel("")
save(fig, "07_tickets")
display(mix.style.format("{:.0%}"))

### Answer to Q3
* **Charger generation is the station attribute that matters.** Gen1 turnaround is ~88 min vs 64 (Gen2) and 42 (Gen3) on normal days. Above 40 °C Gen1 slows to ~174 min and runs out of charged 2W packs in **~58% of hours**, while Gen2/3 barely change (≤1%).
* **Geography is a proxy for Gen1 × heat.** Delhi NCR, Jaipur and Hyderabad have the most Gen1 cabinets *and* the hottest summers, so they carry the summer failure spikes and the highest complaint rates (13–15 tickets per 1,000 swaps vs ~9 in Mumbai, Bengaluru and Pune). Heat-alert days fail at ~16% vs 5%. Rain and floods do not raise failures.
* **Location type, host type and connectivity matter little** once generation and heat are controlled. Connectivity affects *data quality* (missing telemetry, offline sync) rather than service.
* **Complaints follow the mechanism.** Gen1 tickets are dominated by *no battery available*. Gen3 stations (opened later, when the fleet's packs were older) skew towards *low range* and vehicle-performance tickets — a battery problem, not a station problem (Q4).

## 8. Core Question 4 — Battery and equipment performance
*How do SoH, charge cycles, supplier and manufacturing lot relate to delivered range and swap frequency? Do any cohorts stand out?*

In [ ]:
bt["cohort"] = np.where(bt.manufacturing_lot.isin(["KY-2407", "KY-2408", "KY-2409"]), "Kyron KY-2407..09", bt.supplier)
display(pd.crosstab(bt.commission_date.dt.to_period("Q"), bt.supplier).T)
cohort_tbl = bt.groupby(["cohort", "pack_type"]).agg(packs=("battery_id", "size"), purchase_cost=("purchase_cost_inr", "mean"), initial_soh=("initial_soh_pct", "mean"),
                                                     current_soh=("current_soh_pct", "mean"), soh_loss_per_100_swaps=("soh_loss_per_100_swaps", "mean"),
                                                     wear_inr_per_swap=("wear_inr_per_swap", "mean"), retired_share=("retired_date", lambda s: s.notna().mean()))
cohort_tbl.round(2)

In [ ]:
lot = bt.groupby(["supplier", "manufacturing_lot"]).agg(loss=("soh_loss_per_100_swaps", "mean"), packs=("battery_id", "size"),
                                                        commissioned=("commission_date", "min")).reset_index().sort_values("commissioned")
fig, ax = plt.subplots(figsize=(15, 4))
colors = [PAL["bad"] if l in ("KY-2407", "KY-2408", "KY-2409") else {"Cellora": PAL["primary"], "Amptek": PAL["good"], "Kyron": PAL["accent"]}[s]
          for s, l in zip(lot.supplier, lot.manufacturing_lot)]
ax.bar(lot.manufacturing_lot, lot.loss, color=colors)
ax.set_title("SoH points lost per 100 swaps, by manufacturing lot (ordered by commissioning): three Kyron lots degrade ~3× faster")
ax.set_ylabel("SoH pts / 100 swaps"); ax.tick_params(axis="x", rotation=90, labelsize=7)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=PAL["primary"], label="Cellora"), Patch(color=PAL["good"], label="Amptek"), Patch(color=PAL["accent"], label="Kyron (other lots)"),
                   Patch(color=PAL["bad"], label="Kyron KY-2407/08/09")], fontsize=8)
save(fig, "08_lot_degradation")
print(bt.groupby("bms_firmware").soh_loss_per_100_swaps.mean().round(2).to_dict(), "<- BMS firmware makes no difference")

In [ ]:
rng = comp.dropna(subset=["km_valid", "soh_in_pct"])[["battery_in_id", "soh_in_pct", "km_valid", "energy_to_recharge_kwh", "month"]].copy()
rng = rng.merge(bt[["battery_id", "cohort", "pack_type"]].rename(columns={"battery_id": "battery_in_id"}), on="battery_in_id")
rng["soh_band"] = pd.cut(rng.soh_in_pct, [50, 70, 75, 80, 85, 90, 95, 100.01], labels=["<70", "70–75", "75–80", "80–85", "85–90", "90–95", "95–100"])
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, pt in zip(axes[:2], ["2W_2.1kWh", "3W_4.8kWh"]):
    sns.pointplot(data=rng[rng.pack_type == pt], x="soh_band", y="km_valid", hue="cohort", ax=ax, errorbar=None,
                  palette={"Cellora": PAL["primary"], "Amptek": PAL["good"], "Kyron": PAL["accent"], "Kyron KY-2407..09": PAL["bad"]})
    ax.set_title(f"{pt}: km delivered per swap vs SoH"); ax.set_xlabel("SoH of returned pack"); ax.set_ylabel("km since last swap")
    ax.legend(title="cohort (Cellora and Amptek overlap)", fontsize=8, title_fontsize=8)
mt = rng.groupby("month").agg(soh=("soh_in_pct", "mean"), km=("km_valid", "mean"))
mt["bad_lot_share"] = comp.assign(bad=comp.battery_out_id.astype(str).isin(bt.loc[bt.cohort == "Kyron KY-2407..09", "battery_id"])).groupby("month").bad.mean()
ax = axes[2]
ax.plot(mt.index, mt.km, color=PAL["primary"], lw=2.2, label="km per swap"); ax.set_ylabel("km per swap")
ax2 = ax.twinx(); ax2.plot(mt.index, mt.soh, color=PAL["accent"], lw=2.2, label="mean SoH of returned packs"); ax2.grid(False); ax2.set_ylabel("SoH %")
ax.set_title("Network: delivered range fell 30% as the fleet aged"); ax.tick_params(axis="x", rotation=45)
ax.legend(ax.get_lines() + ax2.get_lines(), ["km per swap", "mean SoH of returned packs"], loc="lower left", fontsize=8)
save(fig, "08_range_vs_soh")
mt.iloc[[0, 6, 9, 12, 17]].round(2)

In [ ]:
rdd = comp.groupby(["rider_id", "date"], observed=True).agg(swaps=("event_id", "size"), soh=("soh_out_pct", "mean"))
rdd["soh_band"] = pd.cut(rdd.soh, [50, 70, 75, 80, 85, 90, 95, 100.01], labels=["<70", "70–75", "75–80", "80–85", "85–90", "90–95", "95–100"])
freq = rdd.groupby("soh_band", observed=True).swaps.agg(["mean", "size"]).rename(columns={"mean": "swaps per rider-day", "size": "rider-days"})
freq

### Answer to Q4
* **SoH drives delivered range almost linearly.** A 2W pack returned at 95–100% SoH has covered ~67 km; at <70% SoH, ~38 km. Riders on degraded packs come back sooner: rider-days on 70–80% SoH packs average **~1.55 swaps vs ~1.3** on healthy packs. Degradation therefore *adds load* to the stations that are already stocking out.
* **One cohort stands out: Kyron lots KY-2407, KY-2408 and KY-2409** (1,461 packs commissioned Jul–Sep 2024, the new supplier's first deliveries). They lose **~9.6 SoH points per 100 swaps (2W) and ~7.9 (3W) vs ~3** for every other lot. Every Kyron 2W pack came from these lots. Kyron's later lots (small 3W batches from Oct 2024) degrade normally, which points to a **batch defect** rather than a bad supplier. The 2W packs reached ~61% SoH within a year and were all marked retired on 15 Jun 2025. They make up ~22% of all swaps from October 2024 onwards.
* At the *same* SoH, the bad Kyron lots also deliver less range (2W at 90–100% SoH: ~58 km vs ~66 km for Cellora/Amptek), so riders felt them before SoH showed it.
* **Cost impact:** the bad lots' wear cost is ~₹106 per swap vs ~₹30–33 for 2W packs from other cohorts. This is the September 2024 step in battery wear seen in Q1.
* **BMS firmware makes no difference** (4.3–4.5 SoH pts per 100 swaps across versions). Network-wide, mean SoH of returned packs fell from 99% to 76% and delivered range from ~74 km to ~51 km per swap. Range complaints followed (§11.4).

## 9. Core Question 5 — Pricing and partner economics
*How do the base price change, the peak/off-peak pilot and fleet contract terms relate to swap timing, revenue and margin? Do they affect every segment the same way?*

**What the data says happened** (reconstructed from `list_price_inr` and `tariff_code`, since the brief gives no dates):
* **Base price rise on 1 Jul 2024:** 2W ₹60 → ₹65, 3W ₹100 → ₹110.
* **Peak/off-peak pilot from 1 Oct 2024 in Bengaluru and Pune**, Independent riders see PEAK ₹80 (12–14 h, 19–23 h) and OFFPEAK ₹57 (23–06 h, 14–17 h). Partner riders keep the `PARTNER` tariff code, but the surcharge is added to their employer's invoice where `peak_surcharge_billable = Y`. ZipDrop and Swiggle Go are exempt.

In [ ]:
comp = se[se.completed].copy()
comp["segment"] = np.where(comp.plan_type == "partner_billed", "partner", "independent")
comp["peak_hour"] = comp.hour.isin([12, 13, 19, 20, 21, 22])
print(pd.crosstab(comp.segment, comp.tariff_code))
rm = comp.groupby(["rider_id", "month"], observed=True).size().rename("swaps").reset_index().merge(rd[["rider_id", "plan_type"]], on="rider_id")
pr = rm.pivot_table(index="month", columns="plan_type", values="swaps", aggfunc="mean")
pr.loc["2024-04-01":"2024-10-01"].round(2)

**Base price rise:** no visible demand penalty. Swaps per active rider rose in July for all plan types (the dip in May–June is the summer stockout, not price), and the July 2024 new-rider cohort retained at 88%, above the pre-rise average (§10). Revenue per completed swap rose ~₹5 (+8.6%).

### Peak pilot — difference-in-differences
Treatment: Bengaluru + Pune stations after 1 Oct 2024. Control: the other four cities. Window: Jul–Dec 2024 (after the base-price change, before 2025 expansion). Outcome: share of each station-day's completed swaps that fall in peak hours. Station and date fixed effects; SEs clustered by station.

In [ ]:
fpm = fp.set_index("partner_id").peak_surcharge_billable
comp["exposure"] = np.select([comp.segment == "independent", comp.partner_id.map(fpm).eq("Y")], ["independent (pays PEAK)", "partner, surcharge billed"], "partner, surcharge exempt")
win = comp[(comp.date >= "2024-07-01") & (comp.date < "2025-01-01")].copy()
win["pilot"] = win.city.isin(["Bengaluru", "Pune"]).astype(int)
win["post"] = (win.date >= "2024-10-01").astype(int)
did_rows = []
for grp, g in win.groupby("exposure"):
    sdd = g.groupby(["station_id", "date"], observed=True).agg(peak=("peak_hour", "mean"), n=("event_id", "size"), rev=("amount_charged_inr", "mean"),
                                                           pilot=("pilot", "first"), post=("post", "first")).reset_index()
    sdd["station_id"] = sdd.station_id.astype(str)
    for y in ["peak", "rev"]:
        r = smf.wls(f"{y} ~ pilot:post + C(station_id) + C(date)", data=sdd, weights=sdd.n).fit(cov_type="cluster", cov_kwds={"groups": sdd.station_id})
        did_rows.append({"segment": grp, "outcome": {"peak": "peak-hour share (pp)", "rev": "revenue per swap (₹)"}[y],
                         "DiD estimate": r.params["pilot:post"] * (100 if y == "peak" else 1), "CI low": r.conf_int().loc["pilot:post", 0] * (100 if y == "peak" else 1),
                         "CI high": r.conf_int().loc["pilot:post", 1] * (100 if y == "peak" else 1), "p": r.pvalues["pilot:post"]})
did = pd.DataFrame(did_rows)
did.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
wk = comp[(comp.date >= "2024-06-01") & (comp.date < "2025-03-01") & (comp.segment == "independent")].copy()
wk["group"] = np.where(wk.city.isin(["Bengaluru", "Pune"]), "Pilot cities (BLR, PUN)", "Control cities")
wk["week"] = wk.date.dt.to_period("W").dt.start_time
ws = wk.groupby(["week", "group"]).peak_hour.mean().unstack()
ws.plot(ax=axes[0], color=[PAL["muted"], PAL["accent"]], lw=2.2)
axes[0].axvline(pd.Timestamp("2024-10-01"), color="black", ls=":"); pct(axes[0], decimals=0)
axes[0].set_title("Independent riders: share of swaps in peak hours"); axes[0].set_xlabel("")
peak_fail = se[(se.date >= "2024-07-01") & (se.date < "2025-01-01") & se.hour.isin([12, 13, 19, 20, 21, 22])].copy()
peak_fail["group"] = np.where(peak_fail.city.isin(["Bengaluru", "Pune"]), "Pilot", "Control")
peak_fail["period"] = np.where(peak_fail.date >= "2024-10-01", "After 1 Oct", "Before")
pf = peak_fail.groupby(["group", "period"]).service_failure.mean().unstack()[["Before", "After 1 Oct"]]
pf.plot.bar(ax=axes[1], color=[PAL["muted"], PAL["accent"]], rot=0); pct(axes[1], decimals=1)
axes[1].set_title("Peak-hour failure rate: unchanged by the pilot"); axes[1].set_xlabel("")
save(fig, "09_peak_pilot")
pf.round(4)

**The pilot shifted demand a little, but not where it was needed.**
* Riders whose swaps carry the surcharge moved **2.5 pp (independents) to 3.5 pp (billable partners)** of their swaps out of peak hours (independents: ~60% → ~57%). Exempt partners (ZipDrop, Swiggle Go) did not move (−0.5 pp, not significant).
* Revenue per swap rose **~₹8 for independents and ~₹6 for billable partners** relative to control cities, and not at all for exempt partners. The pilot is **revenue-accretive**.
* **Peak-hour reliability did not change**, because Bengaluru and Pune were already the two most reliable cities (few Gen1 cabinets, mild summers). The pilot tested congestion pricing where there was little congestion to relieve.

### Fleet partner economics

In [ ]:
comp["pid"] = comp.partner_id.fillna("Independent")
pe = comp.groupby("pid").agg(swaps=("event_id", "size"), revenue_m=("amount_charged_inr", lambda s: s.sum() / 1e6), rev_per_swap=("amount_charged_inr", "mean"),
                             discount_per_swap=("discount_inr", "mean"), energy=("energy_cost_inr", "mean"), wear=("wear_inr", "mean"),
                             station=("fixed_per_swap", "mean"), cm1=("cm1", "mean"), cm2=("cm2", "mean"), cm2_total_m=("cm2", lambda s: s.sum() / 1e6),
                             peak_share=("peak_hour", "mean"))
pe = pe.join(fp.set_index("partner_id")[["partner_name", "partner_segment", "contract_type", "discount_pct", "discount_pct_after_amendment", "peak_surcharge_billable", "payment_terms_days"]])
pe["partner_name"] = pe.partner_name.fillna("Independent riders")
pe = pe.sort_values("swaps", ascending=False)
partner_econ = pe.copy()
pe.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
p2 = pe.drop("Independent")
ax = axes[0]
colors = [PAL["bad"] if n == "ZipDrop" else (PAL["accent"] if s == "cargo_3w" else PAL["primary"]) for n, s in zip(p2.partner_name, p2.partner_segment)]
ax.scatter(p2.revenue_m, p2.cm1, s=p2.swaps / 2500, c=colors, alpha=.85, edgecolor="white")
for _, r in p2.iterrows():
    ax.annotate(r.partner_name, (r.revenue_m, r.cm1), fontsize=8, xytext=(5, 3), textcoords="offset points")
ax.set_xlabel("total revenue (₹ million)"); ax.set_ylabel("CM1 per swap (₹, before wear)")
ax.set_title("Largest partner, lowest 2W margin: ZipDrop")
z = comp[comp.pid == "FP-03"].groupby("month").agg(discount=("discount_inr", "mean"), cm1=("cm1", "mean"), swaps=("event_id", "size"))
ax = axes[1]
ax.plot(z.index, z.discount, color=PAL["bad"], lw=2.2, marker="o", ms=3, label="discount per swap (₹)")
ax.plot(z.index, z.cm1, color=PAL["primary"], lw=2.2, marker="o", ms=3, label="CM1 per swap (₹)")
ax.axvline(pd.Timestamp("2024-11-01"), color="black", ls=":"); ax.legend(fontsize=8)
ax.set_title("ZipDrop: the Nov 2024 amendment cut ~₹11 per swap overnight")
save(fig, "09_partners")
zz = comp[comp.pid.isin(["FP-03", "FP-01"]) & comp.month.between("2024-09-01", "2024-12-01")]
zz = zz.pivot_table(index="month", columns="pid", values=["discount_inr", "amount_charged_inr", "cm1"], aggfunc="mean")
zz.columns = [f"{'ZipDrop' if p == 'FP-03' else 'FeastFly'} {m}" for m, p in zz.columns]
print("Oct -> Nov 2024 change in ZipDrop CM1 per swap:", round(zz.loc["2024-11-01", "ZipDrop cm1"] - zz.loc["2024-10-01", "ZipDrop cm1"], 2),
      "| FeastFly (control):", round(zz.loc["2024-11-01", "FeastFly cm1"] - zz.loc["2024-10-01", "FeastFly cm1"], 2))
zz.round(2)

### Answer to Q5
* **Base price rise (Jul 2024):** +₹5 per 2W swap with no detectable volume or retention penalty. It was the single best margin lever in the period, and battery wear absorbed it within two months.
* **Peak/off-peak pilot:** statistically significant but modest demand shifting (~3 pp) and ~₹8 more revenue per exposed swap. **No reliability gain**, because it ran in the two least-congested cities. It affects segments unequally: independents and surcharge-billable partners respond; exempt partners do not.
* **Partner terms decide partner value.** ZipDrop is the **largest partner by volume (549K swaps) but the least profitable 2W partner.** Its Nov 2024 amendment raised its discount from 12% to 28% (≈ ₹7.8 → ₹18.9 per swap). It is exempt from the peak surcharge while doing most of its swaps at peak (see `peak_share`), and it has the longest payment terms (45 days). Between October and November 2024 its revenue and CM1 per swap fell by **~₹11 overnight** (₹59.5 → ₹48.6), while FeastFly and Swiggle Go were flat. Its volume kept growing, so every extra ZipDrop swap now earns ~₹10 less contribution than a FeastFly swap before wear. After wear it is negative. The 3W cargo partners (CargoTuk, HaulKing) earn high revenue per swap but carry ~₹83 of wear, so they have the weakest after-wear margins.

## 10. Core Question 6 — Root cause of new-rider churn

**Definitions**
* **New rider**: signed up on or after 1 Jan 2024 (so their whole history is observed), first attempt before 1 May 2025 (so a 60-day follow-up exists). n ≈ 13.8K.
* **Retained**: at least one swap attempt in days 30–59 after the first attempt. **Churned** = did not come back in that window.
* **Early-experience features** use only the first 14 days (before the outcome window), so they cannot be caused by the churn itself.

In [ ]:
first = se.groupby("rider_id", observed=True).event_ts.min().rename("first_ts")
riders = rd.join(first, on="rider_id")
bad_lot_ids = set(bt.loc[bt.cohort == "Kyron KY-2407..09", "battery_id"])
ev = se[["rider_id", "event_ts", "event_id", "service_failure", "completed", "soh_out_pct", "km_valid", "amount_charged_inr", "tariff_code", "station_id",
         "queue_wait_sec", "battery_out_id"]].merge(riders[["rider_id", "first_ts"]], on="rider_id")
ev["d"] = (ev.event_ts - ev.first_ts).dt.days
new = riders[(riders.signup_date >= "2024-01-01") & (riders.first_ts < "2025-05-01")].copy()
ev = ev[ev.rider_id.isin(new.rider_id)]
new["retained"] = new.rider_id.isin(ev.loc[(ev.d >= 30) & (ev.d < 60), "rider_id"]).astype(int)

e14 = ev[ev.d < 14]
c14 = e14[e14.completed]
f = e14.groupby("rider_id", observed=True).agg(attempts_14d=("event_id", "size"), fail_rate_14d=("service_failure", "mean"))
f = f.join(c14.groupby("rider_id", observed=True).agg(soh_14d=("soh_out_pct", "mean"), km_per_swap_14d=("km_valid", "mean"), price_14d=("amount_charged_inr", "mean"),
                                                    bad_lot_share_14d=("battery_out_id", lambda s: s.astype(str).isin(bad_lot_ids).mean()),
                                                    peak_tariff_share_14d=("tariff_code", lambda s: (s == "PEAK").mean())))
f["first_attempt_failed"] = ev.sort_values("event_ts").groupby("rider_id", observed=True).service_failure.first().astype(int)
home = e14.groupby("rider_id", observed=True).station_id.agg(lambda s: s.astype(str).mode().iloc[0]).rename("home_station")
f = f.join(home)
new = new.merge(f, left_on="rider_id", right_index=True, how="left")
new = new.merge(st[["station_id", "charger_generation", "competitor_within_1_5km_since"]].rename(columns={"station_id": "home_station", "charger_generation": "home_station_gen"}),
                on="home_station", how="left")
new["competitor_nearby"] = (new.competitor_within_1_5km_since.notna() & (new.competitor_within_1_5km_since <= new.first_ts)).astype(int)
t30 = tk.merge(new[["rider_id", "first_ts"]], on="rider_id")
t30 = t30[(t30.created_ts >= t30.first_ts) & (t30.created_ts < t30.first_ts + pd.Timedelta(days=30))]
new = new.merge(t30.groupby("rider_id").size().rename("tickets_30d"), on="rider_id", how="left")
new["tickets_30d"] = new.tickets_30d.fillna(0)
promo = cx[cx.competitor_promo_active][["city", "date"]]
new["competitor_promo_days_30d"] = [((promo.city == c) & (promo.date >= t.normalize()) & (promo.date < t.normalize() + pd.Timedelta(days=30))).sum()
                                    for c, t in zip(new.home_city, new.first_ts)]
new["first_month"] = new.first_ts.dt.to_period("M").dt.to_timestamp()
print(f"New riders: {len(new):,} | 30-59 day retention: {new.retained.mean():.1%}")

In [ ]:
coh = new.groupby("first_month").agg(riders=("rider_id", "size"), retention=("retained", "mean"), early_fail=("fail_rate_14d", "mean"), early_km=("km_per_swap_14d", "mean"))
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(coh.index, coh.retention, color=PAL["primary"], lw=2.5, marker="o", label="30–59 day retention (left)")
pct(ax, decimals=0); ax.set_ylabel("retention"); ax.set_ylim(.74, .93)
ax2 = ax.twinx(); ax2.bar(coh.index, coh.early_fail, width=18, color=PAL["bad"], alpha=.35, label="failure rate in first 14 days (right)")
pct(ax2, decimals=0); ax2.grid(False); ax2.set_ylim(0, .35)
ax.set_title("New-rider cohorts: retention falls exactly when early failures spike")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc="upper center", ncol=2, fontsize=8)
save(fig, "10_cohort_retention")
print("Correlation across monthly cohorts, retention vs early failure:", round(coh.retention.corr(coh.early_fail), 3))
coh.round(3)

In [ ]:
def lift(col, bins=None, labels=None):
    g = new.groupby(pd.cut(new[col], bins, labels=labels) if bins is not None else new[col], observed=True)
    return g.agg(riders=("rider_id", "size"), retention=("retained", "mean"))


tables = {
    "Failure rate, first 14 days": lift("fail_rate_14d", [-.01, 0, .05, .1, .2, 1], ["0%", "0–5%", "5–10%", "10–20%", ">20%"]),
    "First attempt failed": lift("first_attempt_failed"),
    "km per swap, first 14 days": lift("km_per_swap_14d", [0, 45, 55, 65, 75, 500], ["<45", "45–55", "55–65", "65–75", ">75"]),
    "Vehicle class": lift("vehicle_class"), "Plan type": lift("plan_type"), "Signup channel": lift("signup_channel"),
    "Home station generation": lift("home_station_gen"), "Competitor within 1.5 km": lift("competitor_nearby"),
}
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=True)
for ax, (name, t) in zip(axes.flat, tables.items()):
    ax.bar(t.index.astype(str), t.retention, color=PAL["primary"]); ax.set_ylim(.6, .95); pct(ax, decimals=0)
    ax.axhline(new.retained.mean(), color=PAL["bad"], ls=":", lw=1)
    ax.set_title(name, fontsize=10); ax.tick_params(axis="x", rotation=30, labelsize=8)
    for i, (r, n) in enumerate(zip(t.retention, t.riders)):
        ax.text(i, r + .005, f"n={n:,}", ha="center", fontsize=7, color="#555")
fig.suptitle("30–59 day retention by early experience and rider attributes (dotted = overall)", fontweight="bold")
fig.tight_layout()
save(fig, "10_retention_lifts")

### Multivariate model
A logistic regression separates overlapping factors. For example, 3W riders both fail more *and* are a different business. Numeric features are standardised, so each coefficient is the change in log-odds of retention per one standard deviation. `attempts_14d` is included as an **engagement control**, not as a driver: riders who swap more early are mechanically more likely to still be around.

In [ ]:
m = new.dropna(subset=["fail_rate_14d", "km_per_swap_14d", "soh_14d", "price_14d"]).copy()
num = ["fail_rate_14d", "km_per_swap_14d", "soh_14d", "bad_lot_share_14d", "price_14d", "peak_tariff_share_14d", "tickets_30d", "competitor_promo_days_30d", "attempts_14d"]
for c in num:
    m[c + "_z"] = (m[c] - m[c].mean()) / m[c].std()
formula = ("retained ~ " + " + ".join(c + "_z" for c in num) +
           " + first_attempt_failed + competitor_nearby + kyc_verified + C(vehicle_class) + C(plan_type, Treatment('partner_billed'))"
           " + C(signup_channel) + C(home_city, Treatment('Bengaluru')) + C(home_station_gen, Treatment('Gen2'))")
logit = smf.logit(formula, data=m).fit(disp=0)
res = logit.summary2().tables[1].rename(columns={"Coef.": "coef", "P>|z|": "p", "[0.025": "lo", "0.975]": "hi"})[["coef", "lo", "hi", "p"]]
res = res.drop("Intercept")
res["odds_ratio"] = np.exp(res.coef)
print(f"Pseudo R² = {logit.prsquared:.3f}, n = {int(logit.nobs):,}")
res.sort_values("coef").round(3)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

feat_cols = num + ["first_attempt_failed", "competitor_nearby", "kyc_verified", "vehicle_class", "plan_type", "signup_channel", "home_city", "home_station_gen"]
X = pd.get_dummies(m[feat_cols], drop_first=False).astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, m.retained, test_size=.3, random_state=7, stratify=m.retained)
gb = GradientBoostingClassifier(random_state=7, max_depth=3, n_estimators=250, learning_rate=.05).fit(Xtr, ytr)
print(f"Hold-out AUC: {roc_auc_score(yte, gb.predict_proba(Xte)[:, 1]):.3f}")
pi = permutation_importance(gb, Xte, yte, n_repeats=8, random_state=7, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, X.columns)
groups = {c: c for c in num + ["first_attempt_failed", "competitor_nearby", "kyc_verified"]}
for c in X.columns:
    for g in ["vehicle_class", "plan_type", "signup_channel", "home_city", "home_station_gen"]:
        if c.startswith(g + "_"):
            groups[c] = g
imp = imp.groupby(imp.index.map(groups)).sum().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
show = res[~res.index.str.contains("attempts_14d")].sort_values("coef")
ax = axes[0]
ax.errorbar(show.coef, range(len(show)), xerr=[show.coef - show.lo, show.hi - show.coef], fmt="o",
            color=PAL["primary"], ecolor="#9aa5b1", capsize=2)
sig = show.p < .05
ax.scatter(show.coef[sig], np.arange(len(show))[sig.values], color=PAL["bad"], zorder=3, s=25)
ax.set_yticks(range(len(show))); ax.set_yticklabels(show.index.str.replace("C\\(|\\)|Treatment\\('.*?'\\)|, ", "", regex=True).str.replace("_z", " (per SD)"), fontsize=8)
ax.axvline(0, color="black", lw=.8); ax.set_title("Logistic regression: effect on log-odds of retention (red = p<0.05)")
ax = axes[1]
imp.drop("attempts_14d").plot.barh(ax=ax, color=PAL["accent"]); ax.set_title("Gradient boosting: permutation importance (hold-out AUC drop)")
fig.tight_layout()
save(fig, "10_drivers")
imp.sort_values(ascending=False).round(4)

### Answer to Q6 — primary vs secondary drivers

| Tier | Factor | Evidence |
|---|---|---|
| **Primary** | **Early service failure** (stockouts in the first 14 days) | Largest behavioural effect in both models (logit z-statistic ≈ −13; second in permutation importance after the engagement control). Retention falls from ~87% at 0% early failure to ~65% above 20%. A failed *first* attempt alone is associated with ~8 pp lower retention (78% vs 86%). In the model this is absorbed by the 14-day failure rate. Monthly cohort retention tracks the summer failure spikes almost exactly. |
| **Primary** | **Delivered range** (km per swap in the first 14 days) | Second-largest behavioural effect (z-statistic ≈ +9). Riders getting <45 km per swap retain at ~80%. Its effect runs *through* range: bad-lot share and SoH lose significance once range is in the model. |
| Secondary | Vehicle class (3W) | ~3 pp lower retention; strongly negative once failure is controlled, so 3W riders are also less sticky for other reasons (fewer 3W slots, fleet churn). |
| Secondary | Home city (Delhi, Hyderabad, Jaipur, Pune vs Bengaluru) | Significant but smaller. It partly reflects heat and Gen1 exposure beyond the first 14 days. |
| Secondary | Partner-billed plan | Partner-billed riders retain ~2.5 pp less than pay-as-you-go. Their stickiness depends on the employer contract. |
| Minor | Peak-tariff exposure | Small negative effect, significant at p<0.01. Worth watching before any network-wide rollout. |
| **Not a driver** | Competitor within 1.5 km, competitor promo days, signup channel, KYC, home-station generation *per se*, unresolved tickets | Not significant. Raising a ticket is *positively* associated with retention (an engaged rider), so ticket counts are not a churn signal. |

**Bottom line:** new riders leave when the network fails them early, by being empty when they arrive or by handing them a pack that does not last the shift. Both trace back to the two operational causes found in Q2–Q4: Gen1 cabinets in the heat, and degraded (especially bad-lot Kyron) batteries.